## Intermediate Data Science

## Important Information

- Email: [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
- Office Hours take place in Duke 209 - [Office Hours Schedule](https://joannabieri.com/schedule.html)
- [Class Website](https://joannabieri.com/data201_intermediate.html)
- [Syllabus](https://joannabieri.com/data201/IntermediateDataScience.pdf)

## Today's Reading

*Introduction to Machine Learning with Python*, Chapter 3.5.1 - k-Means Clustering.

## Career Reading

*Build a Career in Data Science*, 4.1 and 4.3 Building a Portfolio, 5.1 and 5.2 Finding the Right Job, and 6.1 and 6.2 Resume and Cover Letter Basics. We talk about portfolios, job searching, resumes and cover letters together in class today, so bring your notes!

## Video

Here is an optional video if you want to see the ideas explained another way, this short video (not mine) is really good:

- [StatQuest: K-means clustering](https://www.youtube.com/watch?v=4b5d3muPQmA)

---------------------------------------

## Unsupervised Learning - Clustering

When we did PCA and t-SNE we never gave the computer the labels. We only used the labels at the end, to color the pictures and see if the groups lined up with something real. In the t-SNE picture of the MNIST digits you could see clumps of points, and most of the clumps were one digit.

Today we let the computer find those groups for us. This is called **clustering**: splitting the data into groups (clusters) so that points in the same group are similar to each other and points in different groups are not. There is no label to predict and no right answer to check against, so clustering is **unsupervised** learning.

Why would you want to do this? Here are a few examples:

- A store wants to split its customers into groups so it can treat each group differently (eg. coupons for the bargain hunters, early access for the big spenders).
- A biologist has measurements of hundreds of plants and wants to know if there are different types hiding in the data.
- You have a big pile of documents or pictures and want to sort them before you look at them.

We will learn the most common clustering method, **k-Means**.

In [ ]:
# NOTE - This list of package imports is getting long
# In a professional setting you would only want to
#      import what you need!
# I had chatGPT break the packages into groups here

# ============================================================
# Basic packages
# ============================================================
import os                             # For file and directory operations
import numpy as np                    # For numerical computing and arrays
import pandas as pd                   # For data manipulation and analysis

# ============================================================
# Visualization packages
# ============================================================
import matplotlib.pyplot as plt        # Static 2D plotting
import seaborn as sns                  # Statistical data visualization built on matplotlib

# Interactive visualization with Plotly
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook_connected'   # Set renderer for interactive output in notebooks

# ============================================================
# Scikit-learn: Core utilities for model building and evaluation
# ============================================================
from sklearn.model_selection import train_test_split    # Train/test data splitting
from sklearn.preprocessing import StandardScaler        # Feature scaling

# ============================================================
# Scikit-learn: Data sets
# ============================================================
from sklearn.datasets import make_blobs, make_moons, load_wine

# ============================================================
# Dimensionality Reduction
# ============================================================
from sklearn.decomposition import PCA

# ============================================================
# Clustering - NEW today
# ============================================================
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## k-Means Clustering

**k-Means** splits the data into $k$ clusters. You have to choose $k$, the number of clusters, before you start. Each cluster has a **center** (also called a centroid), which is just the average of all of the points in that cluster. Every point belongs to the cluster with the closest center.

### The Algorithm

Here are the steps:

1. **Choose $k$**, the number of clusters.
2. **Start with $k$ centers.** The computer picks $k$ of the data points at random to be the first centers.
3. **Assign every point to its closest center.** We measure "closest" with the same distance formula we used for KNN:
$$
d(x, c) = \sqrt{(x_1 - c_1)^2 + (x_2 - c_2)^2 + \cdots + (x_n - c_n)^2}
$$
4. **Move each center to the average (the mean) of the points assigned to it.** This is where the "means" in k-Means comes from.
5. **Repeat steps 3 and 4** until the centers stop moving (nobody changes clusters).

### What is it trying to do?

Every time we do steps 3 and 4 the total squared distance from each point to its own center goes down (or stays the same). This total is called the **inertia**:

$$
\text{inertia} = \sum_{i=1}^{n} d(x_i, c_{\text{cluster of } i})^2
$$

So k-Means is looking for $k$ centers that make the inertia as small as possible. Small inertia means that the points are close to their centers, so the clusters are tight.

BEWARE! Because the starting centers are random, you can end up with a different answer if you start in a different place. sklearn deals with this by running the whole algorithm several times from different starting centers and keeping the run with the smallest inertia. That is what `n_init=10` does below.

### A Baby Example - By Hand

Let's do this by hand with six numbers on a number line and $k = 2$:

$$
1, \quad 2, \quad 3, \quad 7, \quad 8, \quad 12
$$

We will start with two BAD centers, $c_A = 1$ and $c_B = 2$. Here we are just using regular distance on the number line, $|x - c|$.

![](images/baby-start.png)

**Round 1**

- Assign: 1 is closest to $c_A$. Everything else (2, 3, 7, 8, 12) is closest to $c_B$.
- Move: $c_A = 1$ and $c_B = \frac{2 + 3 + 7 + 8 + 12}{5} = 6.4$

![](images/baby-round1.png)

**Round 2**

- Assign: 1, 2, and 3 are now closest to $c_A = 1$ (eg. for 3 we get $|3 - 1| = 2$ and $|3 - 6.4| = 3.4$). The points 7, 8, and 12 are closest to $c_B = 6.4$.
- Move: $c_A = \frac{1 + 2 + 3}{3} = 2$ and $c_B = \frac{7 + 8 + 12}{3} = 9$

![](images/baby-round2.png)

**Round 3**

- Assign: 1, 2, 3 still go with $c_A = 2$ and 7, 8, 12 still go with $c_B = 9$. Nobody changed clusters, so we are done!

![](images/baby-round3.png)

The inertia at the end is

$$
(1-2)^2 + (2-2)^2 + (3-2)^2 + (7-9)^2 + (8-9)^2 + (12-9)^2 = 1 + 0 + 1 + 4 + 1 + 9 = 16
$$

Try it yourself: start with $c_A = 7$ and $c_B = 12$ instead. Do you end up with the same clusters?

## Example Data

Here is some made up data where we know there are three groups. `make_blobs` makes round clumps of points. It also gives us labels `y`, but we are going to pretend we don't have them.

In [ ]:
X, y = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)

plt.figure(figsize=(6,5))
plt.scatter(X[:,0], X[:,1], c='gray', alpha=0.6)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Our data, no labels")
plt.grid(True)
plt.show()

Your eyes can probably see three groups right away. Can the computer find them?

### Watch the algorithm work

Before we use sklearn, let's do the steps ourselves so we can see what happens. To make the picture interesting I am going to start with three BAD centers, all on one side of the data. Then we assign and move three times.

In [ ]:
# Three bad starting centers, all on the left side
centers = np.array([[-8.0, 0.0], [-8.0, 2.0], [-8.0, 4.0]])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for step, ax in enumerate(axes):
    # Step 3 - assign each point to the closest center
    distances = np.sqrt(((X[:, None, :] - centers[None, :, :])**2).sum(axis=2))  # distance to each center
    labels = distances.argmin(axis=1)                                          # which center is closest

    ax.scatter(X[:,0], X[:,1], c=labels, cmap='viridis', alpha=0.5)
    ax.scatter(centers[:,0], centers[:,1], c='red', marker='X', s=200)
    ax.set_title(f"Step {step}")

    # Step 4 - move each center to the mean of its points
    centers = np.array([X[labels == j].mean(axis=0) for j in range(3)])
plt.show()

The red X's are the centers. At step 0 they are all on the left side, so the colors don't match the groups (the clump on the right is split between two colors). After one assign and move step the centers are already close to the three clumps. By step 3 they have stopped moving, so we are done.

## k-Means in sklearn

Now let's let sklearn do it. The code looks a lot like every other model we have used, with one BIG difference: there is no `y` in `fit`! Why not? Remember we are doing unsupervised learning - no labels.

In [ ]:
# Define the model - we have to choose the number of clusters
kmeans = KMeans(n_clusters=3,       # k, the number of clusters
                n_init=10,          # run it 10 times from different random starts, keep the best
                random_state=42)    # so we all get the same answer

# Fit the model - notice there is NO y here!
kmeans.fit(X)

# The cluster that each point was put in
print("Cluster labels for the first 10 points:", kmeans.labels_[:10])

After we fit the model it saves a few things for us:

- `kmeans.labels_` is the cluster number of every point.
- `kmeans.cluster_centers_` are the centers.
- `kmeans.inertia_` is the total squared distance from each point to its center.

In [ ]:
print("Centers:")
print(kmeans.cluster_centers_)
print("Inertia:", kmeans.inertia_)

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(X[:,0], X[:,1], c=kmeans.labels_, cmap='viridis', alpha=0.6)
plt.scatter(kmeans.cluster_centers_[:,0], kmeans.cluster_centers_[:,1], c='red', marker='X', s=200)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("k-Means with k = 3")
plt.grid(True)
plt.show()

We can also put new points into clusters with `predict`. The model just finds the closest center.

In [ ]:
new_points = np.array([[0, 0],
                       [-6, -6]])
kmeans.predict(new_points)

In [ ]:
new_clusters = kmeans.predict(new_points)

plt.figure(figsize=(6,5))
plt.scatter(X[:,0], X[:,1], c=kmeans.labels_, cmap='viridis', alpha=0.3)
plt.scatter(kmeans.cluster_centers_[:,0], kmeans.cluster_centers_[:,1], c='red', marker='X', s=200, label='centers')
plt.scatter(new_points[:,0], new_points[:,1], c=new_clusters, cmap='viridis', vmin=0, vmax=2,
            marker='*', s=500, edgecolors='black', label='new points')
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("New points go to the closest center")
plt.legend()
plt.grid(True)
plt.show()


NOTE - the cluster numbers are just names! The computer called one group 0, one group 1, and one group 2, but there is nothing special about those numbers. If you change the `random_state` the same group could be called 2 instead of 0.

## Choosing k

In our example we knew there were three groups. In real data you don't know! So how do we choose $k$?

### The Elbow Method

One idea is to look at the inertia. More clusters always means points are closer to their centers, so the inertia ALWAYS goes down as $k$ goes up. If $k$ is the same as the number of points, every point is its own center and the inertia is 0. So we can't just pick the smallest inertia. This would be overfitting!

Instead we plot the inertia for a bunch of values of $k$ and look for the **elbow**: the place where adding another cluster stops helping very much.

In [ ]:
inertias = []
k_values = range(1, 11)
for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(6,4))
plt.plot(k_values, inertias, 'o-')
plt.xlabel("k (number of clusters)")
plt.ylabel("Inertia")
plt.title("The Elbow Method")
plt.grid(True)
plt.show()

The inertia drops A LOT going from 1 to 2 to 3 clusters, and then it barely changes. The elbow is at $k = 3$. This is point where we gain very little by making the model more complicated.

### The Silhouette Score

The elbow is not always easy to see. Another measure is the **silhouette score**. For each point we compute two numbers:

- $a$ = the average distance from the point to the other points in its OWN cluster (how tight is my cluster?)
- $b$ = the average distance from the point to the points in the NEAREST OTHER cluster (how far away is the next group?)

Then the silhouette for that point is

$$
s = \frac{b - a}{\max(a, b)}
$$

- $s$ close to 1 means the point is much closer to its own cluster than to the next one. Great!
- $s$ close to 0 means the point is right on the border between two clusters.
- $s$ less than 0 means the point is closer to a different cluster than its own. Probably in the wrong cluster.

The silhouette score is the average of $s$ over all the points. Bigger is better. It needs at least 2 clusters.

In [ ]:
scores = []
k_values = range(2, 11)
for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)                  # fit and give me the labels in one step
    scores.append(silhouette_score(X, labels))  # NO y here either, just the data and the clusters

plt.figure(figsize=(6,4))
plt.plot(k_values, scores, 'o-')
plt.xlabel("k (number of clusters)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score")
plt.grid(True)
plt.show()

print("Best k:", k_values[np.argmax(scores)], "with silhouette score", round(max(scores), 3))

Both methods agree, $k = 3$.

NOTE - these are tools to help you choose, not rules. In real data the elbow might be soft and the silhouette scores might be close together. Then you also think about what the clusters MEAN. Three customer groups that a marketing team can actually use might be better than seven groups with a slightly higher score. 

How do you figure out what the clusters mean? Well after clustering you can try coloring the points by categories that they represent and see if there are any clear separations.

## You Try

Make new data with `make_blobs(n_samples=500, centers=5, cluster_std=1.0, random_state=7)` and pretend you don't know how many groups there are. Make the elbow plot and the silhouette plot. Which $k$ would you choose? Do the two methods agree? Plot the data colored by your clusters.

---------------------------------------

## k-Means NEEDS Scaling

k-Means uses distances, just like KNN and SVMs. So if one feature has much bigger numbers than the others, it takes over the distance and the other features don't matter.

Let's go back to the Wine data. There are 13 chemical measurements of wines from 3 cultivars (types of grapes). We will cluster WITHOUT the labels and then, at the very end, compare our clusters to the cultivars to see if k-Means found something real.

In [ ]:
wine = load_wine(as_frame=True)
X_wine = wine.data
y_wine = wine.target

# Look at how different the ranges are
X_wine.describe().loc[['min', 'max']].T.sort_values('max')

Look at `proline`, it goes all the way up to 1680, but `hue` is never bigger than 2. Let's try k-Means with 3 clusters with and without scaling.

In [ ]:
# No scaling
km_raw = KMeans(n_clusters=3, n_init=10, random_state=42)
clusters_raw = km_raw.fit_predict(X_wine)

# With scaling
X_wine_scaled = StandardScaler().fit_transform(X_wine)
km_scaled = KMeans(n_clusters=3, n_init=10, random_state=42)
clusters_scaled = km_scaled.fit_predict(X_wine_scaled)

Now we use the labels, ONLY to check. Wine has 13 features, so we can't plot it directly. Let's use PCA from last time to squish the scaled data down to 2D, just so we can draw it. Then we color the same picture three ways: by the real cultivar, by the clusters WITHOUT scaling, and by the clusters WITH scaling.


In [ ]:
# PCA down to 2 dimensions, just so we can draw it
X_wine_2d = PCA(n_components=2).fit_transform(X_wine_scaled)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(X_wine_2d[:,0], X_wine_2d[:,1], c=y_wine, cmap='viridis')
axes[0].set_title("The real cultivars")
axes[1].scatter(X_wine_2d[:,0], X_wine_2d[:,1], c=clusters_raw, cmap='viridis')
axes[1].set_title("k-Means, NO scaling")
axes[2].scatter(X_wine_2d[:,0], X_wine_2d[:,1], c=clusters_scaled, cmap='viridis')
axes[2].set_title("k-Means, WITH scaling")
for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
plt.show()


Remember the cluster numbers are just names, so the colors might not match from picture to picture. Look at the GROUPS instead. Without scaling the clusters are mostly about proline, so the groups cut right across the cultivars. With scaling the three groups look almost exactly like the real cultivars. Only 6 of the 178 wines end up in a cluster with a different cultivar!

So the rule is the same as for KNN and SVMs: **scale your data before k-Means.**

## You Try

Run k-Means on the scaled wine data with $k = 2, 3, 4, 5, 6$ and compute the silhouette score for each. Which $k$ does the silhouette score like best? Does it match the number of cultivars? Make the elbow plot too. Is the elbow easy to see with real data?

---------------------------------------

## When k-Means Fails

Each point goes to its closest center. This means that the border between two clusters is always a straight line halfway between the two centers. So k-Means works best when the clusters are round blobs of about the same size. What happens if the groups have a different shape?

In [ ]:
X_moons, y_moons = make_moons(n_samples=200, noise=0.05, random_state=42)

km_moons = KMeans(n_clusters=2, n_init=10, random_state=42)
moon_clusters = km_moons.fit_predict(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(X_moons[:,0], X_moons[:,1], c=y_moons, cmap='coolwarm')
axes[0].set_title("The two real groups")
axes[1].scatter(X_moons[:,0], X_moons[:,1], c=moon_clusters, cmap='coolwarm')
axes[1].scatter(km_moons.cluster_centers_[:,0], km_moons.cluster_centers_[:,1], c='black', marker='X', s=200)
axes[1].set_title("k-Means with k = 2")
plt.show()

k-Means cut the moons with a straight line! It can't follow the curved shapes, because it only knows "which center am I closest to?". Each cluster gets the tip of the other moon.

NOTE - there are other clustering methods that can handle shapes like this. The book talks about agglomerative clustering and DBSCAN in 3.5.2 and 3.5.3 if you are curious. We will stick with k-Means.


---------------------------------------

## Comparing k-Means and KNN

These two get mixed up a lot, because they both have a $k$ and they both use distance. But they do very different things.

| | k-Nearest Neighbors (KNN) | k-Means |
|----------|----------------|------------------------|
| **Type** | Supervised, needs labels | Unsupervised, no labels |
| **What it does** | Predicts the label of a new point | Splits the data into groups |
| **What k means** | Number of neighbors that vote | Number of clusters |
| **Fit** | `fit(X, y)` | `fit(X)` |
| **How do we know it is good?** | Accuracy (or MSE) on a validation/test set | Inertia, silhouette score, and does it MEAN something? |
| **Scaling** | Needed | Needed |

---------------------------------------

## Homework 19

The full assignment is in `HW_day19.ipynb` in your sandbox. You will cluster the customers of a shopping mall into groups using k-Means, decide how many groups to use, and describe each group so that a marketing team could use your results.

Work the problems in your sandbox and push it, your sandbox is your backup. Your write-up notebook goes in your team's repo, along with Homework 18. Check the class website for the due date.